# Conditioning captions

The grid pairs real MOJAVE observations with eleven synthetic examples. Real panels show the restoring beam in the lower-left corner when beam metadata is available. Each panel key shows its conditioning and estimate values alongside the overlay glyphs. The footer explains the glyphs and series colors.

In [ ]:
from io import BytesIO

import matplotlib as mpl
import numpy as np
from IPython.display import Image, display
from matplotlib.lines import Line2D

from radio_diffusion.utils import plotting
from radio_diffusion.utils.plotting import ImageOverlay, OverlayGlyph

## Plot settings

Set `MOJAVE_SOURCE_COUNT` to choose how many real FITS observations to load. The notebook samples files at random and can include multiple epochs of the same source. Set the count to `0` for a synthetic-only grid. `SHOW_PLAIN_GRID` also displays the same calibrated views without annotations.

In [ ]:
MOJAVE_SOURCE_COUNT = 6
SHOW_AXES = True
SHARED_COLORBAR = True
SHOW_PLAIN_GRID = False
ANNOTATED_PANELS = None  # All panels, or a set of sample IDs.
VIEW_MAS = 4.2
INTENSITY_LIMITS = (-0.001, 0.56)
FADE_EXTENT_ARCS = True

SERIES = {
    "conditioning": ("Conditioning", {"color": "#259BC4", "linestyle": "dashed"}),
    "estimate": ("Image estimate", {"color": "#D47E16", "linestyle": "solid"}),
}
RIDGE_STYLE = {"color": "#C98DE1", "linewidth": 0.9, "alpha": 0.85}

## Example data

The synthetic and real samples each supply an image, coordinate calibration, jet measurements, and a ridgeline. The real-sample diagnostics also show the FITS filename, restoring beam, and noise estimate.

In [ ]:
samples = [
    dict(id="S01", source="J0813+2910", image_pa=121, image_extent=2.85, peak=0.44,
         annotations={"conditioning": (106, 2.55), "estimate": (121, 2.85)}),
    dict(id="S02", source="J1126−0427", image_pa=66, image_extent=1.60, peak=0.18,
         annotations={"conditioning": (66, 1.60)}),
    dict(id="S03", source="J1502+3741", image_pa=295, image_extent=2.40, peak=0.56,
         annotations={"estimate": (295, 2.40)}),
    dict(id="S04", source="J2034−1152", image_pa=205, image_extent=2.25, peak=0.29,
         annotations={"conditioning": (193, None), "estimate": (205, 2.25)}),
]

SIZE = 88
REFERENCE = (SIZE - 1) / 2
PIXEL_SCALE_MAS = 0.086
coordinates = plotting.mojave_coordinates({
    "reference_pixel_xy": [REFERENCE, REFERENCE],
    "pixel_to_east_north_mas": [[-PIXEL_SCALE_MAS, 0], [0, PIXEL_SCALE_MAS]],
})


def demo_image(sample, seed):
    row, column = np.indices((SIZE, SIZE))
    east, north = coordinates.from_pixels(np.stack((column, row), axis=-1)).transpose(2, 0, 1)
    pa = np.deg2rad(sample["image_pa"])
    along = east * np.sin(pa) + north * np.cos(pa)
    across = east * np.cos(pa) - north * np.sin(pa)
    extent = sample["image_extent"]
    forward = np.maximum(along, 0)
    bend = 0.035 * forward**2
    width = 0.105 + 0.045 * forward
    core = np.exp(-0.5 * ((along / 0.20)**2 + (across / 0.14)**2))
    jet = 0.28 * np.exp(-forward / 1.5 - 0.5 * ((across - bend) / width)**2)
    jet *= (along >= 0) * np.clip((extent - along) / 0.42, 0, 1)
    knot = 0.095 * np.exp(
        -0.5 * (((along - extent * 0.63) / 0.22)**2 + ((across - bend) / 0.14)**2)
    )
    noise = np.random.default_rng(seed).normal(0, 0.00025, core.shape)
    return sample["peak"] * (core + jet + knot) + noise


for index in range(5, 12):
    pa = (40 + index * 37) % 360
    extent = 1.45 + 0.19 * (index % 8)
    samples.append(dict(
        id=f"S{index:02d}", source=f"Synthetic {index:02d}",
        image_pa=pa, image_extent=extent, peak=0.17 + 0.07 * (index % 5),
        annotations={"conditioning": ((pa - 11) % 360, extent + 0.25), "estimate": (pa, extent)},
    ))


def synthetic_ridge(sample):
    along = np.linspace(0.15, sample["image_extent"], 17)
    pa = np.deg2rad(sample["image_pa"])
    forward = np.array([np.sin(pa), np.cos(pa)])
    across = np.array([np.cos(pa), -np.sin(pa)])
    return along[:, None] * forward + (0.035 * along**2)[:, None] * across


synthetic_samples = samples
for index, sample in enumerate(synthetic_samples):
    sample["image"] = demo_image(sample, seed=40 + index)
    sample["coordinates"] = coordinates
    sample["ridge"] = synthetic_ridge(sample)

In [ ]:
import warnings
from pathlib import Path
from random import shuffle

from astropy import units as u
from astropy.io import fits
from astropy.wcs import FITSFixedWarning

from radio_diffusion.data.mojave_metadata import (
    JetOrientationSettings, crop_spatial_geometry, get_noise_jy_per_beam,
    jet_orientation_deg, trace_jet_ridgeline,
)
from radio_diffusion.data.preprocess_mojave import PREPROCESSING_SETTINGS
from radio_diffusion.paths import get_path


REAL_EXAMPLE_SERIES = "estimate"


def read_mojave_example(path: Path, sample_id: str) -> dict:
    """Load a native-beam observation and measure its jet."""
    orientation = JetOrientationSettings(inner_mas=1.0, outer_mas=3.0, sigma_cut=3.0)
    with fits.open(path, memmap=False) as hdul, warnings.catch_warnings():
        warnings.simplefilter("ignore", FITSFixedWarning)
        image_hdu = hdul[0]
        header = image_hdu.header
        image = np.asarray(np.squeeze(image_hdu.data), dtype=np.float32).copy()
        frame = plotting.mojave_coordinates(crop_spatial_geometry(header, left=0, top=0))
        noise = get_noise_jy_per_beam(image_hdu, "four_corner")

        beam_header = np.asarray(
            [header.get(key, np.nan) for key in ("BMAJ", "BMIN", "BPA")], dtype=float,
        )
        major_deg, minor_deg, beam_pa = beam_header
        beam = None
        if np.isfinite(beam_header).all() and 0 < minor_deg <= major_deg:
            major_mas, minor_mas = (beam_header[:2] * u.deg).to_value(u.mas)
            beam = (major_mas, minor_mas, beam_pa)

        pa = np.nan
        if "AIPS CC" in hdul:
            pa = jet_orientation_deg(image_hdu, hdul["AIPS CC"], noise, orientation)
        ridge = []
        ridge_stop = None
        if np.isfinite(pa):
            ridge, ridge_stop = trace_jet_ridgeline(
                image_hdu, noise, pa, PREPROCESSING_SETTINGS.ridgeline,
            )
        ridge_pixels = [(point["x_pixel"], point["y_pixel"]) for point in ridge]
        return dict(
            id=sample_id,
            source=path.name.split(".", 1)[0],
            filename=path.name,
            image=image,
            coordinates=frame,
            beam=beam,
            peak_jy_per_beam=np.nanmax(image),
            noise_jy_per_beam=noise,
            ridge=frame.from_pixels(ridge_pixels) if ridge else np.empty((0, 2)),
            ridge_stop=ridge_stop,
            annotations={
                REAL_EXAMPLE_SERIES: (
                    pa if np.isfinite(pa) else None,
                    ridge[-1]["radius_mas"] if ridge else None,
                ),
            },
        )


real_samples = []
if MOJAVE_SOURCE_COUNT > 0:
    fits_dir = get_path("MOJAVE_FITS_DIR")
    paths = [*fits_dir.glob("*.icn.fits"), *fits_dir.glob("*.icn.fits.gz")]
    shuffle(paths)
    for path in paths:
        try:
            sample = read_mojave_example(path, f"R{len(real_samples) + 1:02d}")
        except (OSError, KeyError, ValueError) as error:
            print(f"Skipping {path.name}: {error}")
            continue
        real_samples.append(sample)
        if len(real_samples) == MOJAVE_SOURCE_COUNT:
            break

    if len(real_samples) < MOJAVE_SOURCE_COUNT:
        print(f"Loaded {len(real_samples)} of {MOJAVE_SOURCE_COUNT} requested MOJAVE observations.")

samples = [*real_samples, *synthetic_samples]

In [ ]:
# Diagnostics for the real MOJAVE panels
if not real_samples:
    print("The grid has only synthetic samples.")

for sample in real_samples:
    rows, columns = sample["image"].shape
    pa, extent = sample["annotations"][REAL_EXAMPLE_SERIES]
    pa_text = "—" if pa is None else f"{pa:.1f}°"
    extent_text = "—" if extent is None else f"{extent:.2f} mas"

    print(f"{sample['id']} | {sample['source']} | {sample['filename']}")
    print(
        f"  Image {columns} × {rows} px, peak {sample['peak_jy_per_beam']:.4g} Jy/beam, "
        f"RMS noise {sample['noise_jy_per_beam']:.4g} Jy/beam"
    )
    beam = sample["beam"]
    if beam is None:
        print("  Restoring beam unavailable")
    else:
        major_mas, minor_mas, beam_pa = beam
        print(f"  Restoring beam {major_mas:.3f} × {minor_mas:.3f} mas, PA {beam_pa:.1f}°")
    print(f"  Jet PA {pa_text}, radial extent {extent_text}, ridgeline {len(sample['ridge'])} points")
    if sample["ridge_stop"] is not None:
        print(f"  Ridgeline stopping reason {sample['ridge_stop']}")
    print()

## Overlays and panel keys

The overlays use the sample's coordinates in milliarcseconds. A panel key reuses the glyph of each drawn mark. When a mark is unavailable, the key can use a standalone glyph alongside the available value or a dash.

In [ ]:
def prepare_annotations(sample, *, show_glyph: bool = True):
    overlay = ImageOverlay(coordinates=sample["coordinates"])
    if ANNOTATED_PANELS is not None and sample["id"] not in ANNOTATED_PANELS:
        return overlay, []

    items = []
    beam = sample.get("beam")
    if beam is not None:
        major_mas, minor_mas, beam_pa = beam
        items.append(plotting.restoring_beam(
            *beam,
            label=f"Restoring beam {major_mas:.3f} × {minor_mas:.3f} mas, PA {beam_pa:.1f}°",
            show_glyph=show_glyph,
        ))
    angle_entries = []
    extent_entries = []
    for series, (series_label, style) in SERIES.items():
        pa, extent = sample["annotations"].get(series, (None, None))
        pa = pa if pa is not None and np.isfinite(pa) else None
        extent = extent if extent is not None and np.isfinite(extent) and extent > 0 else None

        arrow_style = {**style, "linewidth": 1.25}
        if pa is None:
            arrow_glyph = OverlayGlyph.from_shape("arrow", **arrow_style)
        else:
            angle = np.deg2rad(pa)
            tip = (1.65 * np.sin(angle), 1.65 * np.cos(angle))
            arrow = ImageOverlay.arrow(
                (0, 0), tip, label=f"{series_label} jet position angle {pa:.1f}°",
                show_glyph=show_glyph, **arrow_style,
            )
            items.append(arrow)
            arrow_glyph = arrow.glyph
        angle_entries.append((arrow_glyph, "—" if pa is None else f"{pa:.0f}°"))

        arc_style = dict(arrow_style)
        if FADE_EXTENT_ARCS:
            arc_style["linestyle"] = "solid"
        if pa is not None and extent is not None:
            arc = ImageOverlay.arc(
                (0, 0), extent, angle_deg=90 - pa, delta_angle_deg=24,
                fade=FADE_EXTENT_ARCS, label=f"{series_label} radial extent {extent:.2f} mas",
                show_glyph=show_glyph, **arc_style,
            )
            items.append(arc)
            arc_glyph = arc.glyph
        else:
            arc_glyph = OverlayGlyph.from_shape("arc", fade=FADE_EXTENT_ARCS, **arc_style)
        extent_entries.append((arc_glyph, "—" if extent is None else f"{extent:.2f} mas"))

    ridge = sample["ridge"]
    if len(ridge) >= 2:
        ridgeline = ImageOverlay.polyline(
            ridge, smooth=True, label=f"Ridgeline with {len(ridge)} points",
            show_glyph=show_glyph, **RIDGE_STYLE,
        )
        items.append(ridgeline)
        ridge_entry = (ridgeline.glyph, f"{len(ridge)} pts")
    else:
        ridge_entry = (OverlayGlyph.from_shape("polyline", **RIDGE_STYLE), "—")

    overlay.append(items)
    # Matplotlib fills legend columns from top to bottom.
    return overlay, [*angle_entries, ridge_entry, *extent_entries]

In [ ]:
images = [sample["image"] for sample in samples]
columns = 4
rows = (len(images) + columns - 1) // columns
grid_options = dict(
    nrow=columns,
    sample_labels=[f"{sample['id']}  |  {sample['source']}" for sample in samples],
    figsize=(14, 3.3 * rows),
    sharex=True,
    sharey=True,
    show_axes=SHOW_AXES,
    colorbar="shared" if SHARED_COLORBAR else "each",
    colorbar_label="Jy/beam",
)
if SHARED_COLORBAR:
    grid_options["vmin"], grid_options["vmax"] = INTENSITY_LIMITS


def display_grid(figure):
    png = BytesIO()
    figure.savefig(png, format="png", dpi=120)
    display(Image(data=png.getvalue()))


def show_grid(*, annotated: bool):
    figure, axes = plotting.plot_grid(images, **grid_options)
    panels = list(axes.flat)[:len(samples)]
    has_panel_keys = False
    has_restoring_beam = False

    for ax, sample in zip(panels, samples):
        if annotated:
            overlay, entries = prepare_annotations(sample)
            overlay.overlay(ax)
            if entries:
                handles, labels = zip(*entries)
                plotting.overlay_legend(
                    ax, handles, labels, loc="lower right", ncols=2,
                    fontsize=8, labelcolor="white", facecolor="#172031",
                    edgecolor="#99a5b6", framealpha=0.92, handlelength=1.6,
                    handletextpad=0.3, columnspacing=0.75, labelspacing=0.25,
                )
                has_panel_keys = True
                has_restoring_beam |= sample.get("beam") is not None
        else:
            sample["coordinates"].apply(ax)
        ax.label_outer()

    # Set the shared view after calibrating all panels.
    panels[0].set(xlim=(VIEW_MAS, -VIEW_MAS), ylim=(-VIEW_MAS, VIEW_MAS))

    if has_panel_keys:
        meanings = {"arrow": "Jet PA", "arc": "Radial extent", "polyline": "Ridgeline"}
        if has_restoring_beam:
            meanings["ellipse"] = "Restoring beam"
        handles = [
            OverlayGlyph.from_shape(shape, color="#42536a", linewidth=1.25)
            for shape in meanings
        ]
        labels = list(meanings.values())
        for label, style in SERIES.values():
            handles.append(Line2D([], [], linewidth=1.4, **style))
            labels.append(label)
        plotting.overlay_legend(
            figure, handles, labels, loc="outside lower center", ncols=len(handles),
            frameon=False, handlelength=2.6,
        )

    display_grid(figure)


with mpl.rc_context({"font.size": 9, "axes.labelsize": 9, "figure.facecolor": "white"}):
    show_grid(annotated=True)
    if SHOW_PLAIN_GRID:
        show_grid(annotated=False)

In [ ]:
# Individual images with a text-only legend
examples = [real_samples[0], synthetic_samples[0]] if real_samples else synthetic_samples[:2]
vmin, vmax = INTENSITY_LIMITS if SHARED_COLORBAR else (None, None)

for sample in examples:
    ax, _ = plotting.plot_image(
        sample["image"], show_axes=SHOW_AXES, colorbar=True, colorbar_label="Jy/beam",
        vmin=vmin, vmax=vmax,
    )
    overlay, _ = prepare_annotations(sample, show_glyph=False)
    overlay.overlay(ax)
    ax.set(
        title=f"{sample['id']}  |  {sample['source']}",
        xlim=(VIEW_MAS, -VIEW_MAS), ylim=(-VIEW_MAS, VIEW_MAS),
    )
    if overlay:
        overlay.legend(
            ax, loc="upper right", handlelength=0, handletextpad=0, fontsize=8,
            frameon=True, labelcolor="white", facecolor="#172031",
            edgecolor="#99a5b6", framealpha=0.92,
        )
    ax.figure.set_size_inches(7, 6)
    display_grid(ax.figure)

## Brightness contour preview

Each Stokes I pixel stores the total brightness at that position. The contours use these pixel values before the display stretch. The real MOJAVE panel starts at 3.5 times its measured background RMS, and each subsequent level doubles.

In [ ]:
def draw_brightness_contours(ax, image, coordinates, noise_rms: float) -> list[float]:
    """Draw doubling brightness contours in the image's calibrated coordinates."""
    valid = image[np.isfinite(image)]
    if valid.size == 0 or not np.isfinite(noise_rms) or noise_rms <= 0:
        return []

    floor, peak = valid.min(), valid.max()
    levels = []
    level = 3.5 * noise_rms
    while level < peak:
        if level > floor:
            levels.append(level)
        level *= 2

    if levels:
        rows, columns = np.indices(image.shape)
        east, north = coordinates.from_pixels(
            np.stack((columns, rows), axis=-1),
        ).transpose(2, 0, 1)
        ax.contour(east, north, image, levels=levels, colors="white", linewidths=0.8)
    return levels


if real_samples:
    sample = real_samples[0]
    vmin, vmax = INTENSITY_LIMITS if SHARED_COLORBAR else (None, None)
    ax, _ = plotting.plot_image(
        sample["image"], show_axes=SHOW_AXES, colorbar=True, colorbar_label="Jy/beam",
        vmin=vmin, vmax=vmax,
    )
    coordinates = sample["coordinates"]
    coordinates.apply(ax)
    noise_rms = sample["noise_jy_per_beam"]
    levels = draw_brightness_contours(ax, sample["image"], coordinates, noise_rms)
    levels_text = ", ".join(f"{level:.3g}" for level in levels) or "none"
    print(f"{sample['id']} | RMS {noise_rms:.3g} Jy/beam | contour levels {levels_text}")
    ax.set(
        title=f"{sample['id']}  |  {sample['source']}",
        xlim=(VIEW_MAS, -VIEW_MAS), ylim=(-VIEW_MAS, VIEW_MAS),
    )
    ax.figure.set_size_inches(7, 6)
    display_grid(ax.figure)
else:
    print("Load a real MOJAVE observation to preview brightness contours.")

In [ ]:
# Stokes I contour grid for the loaded MOJAVE observations
if real_samples:
    contour_columns = 3
    contour_rows = (len(real_samples) + contour_columns - 1) // contour_columns
    contour_grid_options = dict(
        grid_options,
        nrow=contour_columns,
        figsize=(12, 3.4 * contour_rows),
        sample_labels=[f"{sample['id']}  |  {sample['source']}" for sample in real_samples],
    )
    figure, axes = plotting.plot_grid(
        [sample["image"] for sample in real_samples], **contour_grid_options,
    )
    for ax, sample in zip(axes.flat, real_samples):
        coordinates = sample["coordinates"]
        coordinates.apply(ax)
        noise_rms = sample["noise_jy_per_beam"]
        levels = draw_brightness_contours(ax, sample["image"], coordinates, noise_rms)
        levels_text = ", ".join(f"{level:.3g}" for level in levels) or "none"
        print(f"{sample['id']} | RMS {noise_rms:.3g} Jy/beam | contour levels {levels_text}")
        ax.label_outer()

    axes.flat[0].set(xlim=(VIEW_MAS, -VIEW_MAS), ylim=(-VIEW_MAS, VIEW_MAS))
    display_grid(figure)
else:
    print("Load a real MOJAVE observation to show the contour grid.")

In [ ]:
# One real observation with four independently scaled colorbars
from random import choice

from matplotlib.figure import Figure

if real_samples:
    sample = choice(real_samples)
    scales = [
        ("Linear", mpl.colors.Normalize()),
        ("Square root", mpl.colors.PowerNorm(gamma=0.5)),
        ("Logarithmic", mpl.colors.LogNorm()),
        ("Asinh", mpl.colors.AsinhNorm(linear_width=0.001)),
    ]

    figure = Figure(figsize=(11, 9), layout="constrained")
    figure.suptitle(sample["filename"])
    axes = figure.subplots(2, 2, sharex=True, sharey=True)
    for ax, (scale_label, norm) in zip(axes.flat, scales):
        plotting.plot_image(
            sample["image"], ax=ax, norm=norm, show_axes=SHOW_AXES,
            colorbar=True, colorbar_label="Jy/beam",
        )
        sample["coordinates"].apply(ax)
        ax.set_title(scale_label)
        ax.label_outer()

    axes[0, 0].set(xlim=(VIEW_MAS, -VIEW_MAS), ylim=(-VIEW_MAS, VIEW_MAS))
    display_grid(figure)
else:
    print("Load a real MOJAVE observation to compare colorbar scales.")